# Colab — HDFS + BGL full + Dual Experts + Fusion + LR/SVM/RF/XGB (Class Weights, V2 Strict)

Cette version garde le même pipeline que la version précédente, mais remplace l'équilibrage par sous-échantillonnage par une gestion du déséquilibre via **class weights**.

- LR: `class_weight="balanced"`
- SVM: `class_weight="balanced"`
- RF: `class_weight="balanced_subsample"`
- XGBoost: `scale_pos_weight` calculé sur le train


Amélioration méthodologique: on conserve le pipeline, mais on l’emploie dans une version plus rigoureuse pour l’évaluation avec gestion du déséquilibre au niveau des modèles.

In [ ]:
# Colab — HDFS + BGL full + Dual Experts + Fusion + LR/SVM/RF/XGB (Class Weights, V2 Strict)

# Dans Colab, exécute d'abord:
# !pip -q install drain3 scikit-learn matplotlib pandas numpy scipy tqdm xgboost

import os
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT_DIR = "/content"
HDFS_LOG_NAME = "HDFS.log"
HDFS_LABEL_NAME = "anomaly_label.csv"
BGL_LOG_NAME = "BGL.log"

HDFS_SAMPLE_FRAC = 1.0
BGL_SAMPLE_FRAC = 1.0
BGL_WINDOW_SIZE = 40
BGL_STRIDE = 20
TEST_SIZE = 0.20

TFIDF_MAX_FEATURES = 5000
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 2
TEXT_SVD_COMPONENTS = 256
N_SPLITS_OOF = 5

print("Configuration chargée — version class weights V2 stricte (split d'abord, poids de classes sur train)")


def find_file(root_dir, filename):
    root = Path(root_dir)
    matches = list(root.rglob(filename))
    if not matches:
        return None
    matches = sorted(matches, key=lambda p: (len(str(p)), str(p)))
    return str(matches[0])


def safe_sample_frac(df, frac, label_col="label", seed=42):
    if frac >= 1.0:
        return df.copy().reset_index(drop=True)
    n_total = len(df)
    n_target = max(2, int(n_total * frac))
    y = df[label_col].values
    if len(np.unique(y)) < 2:
        return df.sample(n=min(n_target, n_total), random_state=seed).reset_index(drop=True)
    try:
        sampled_idx, _ = train_test_split(
            df.index, train_size=n_target, stratify=y, random_state=seed
        )
        return df.loc[sampled_idx].reset_index(drop=True)
    except Exception:
        return df.sample(n=min(n_target, n_total), random_state=seed).reset_index(drop=True)


def print_distribution(df, name, label_col="label"):
    print(f"
[{name}] shape={df.shape}")
    print(df[label_col].value_counts(dropna=False).sort_index())


def compute_scale_pos_weight(y):
    y = np.asarray(y).astype(int)
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    if n_pos == 0:
        return 1.0
    return max(1.0, n_neg / max(n_pos, 1))


def extract_block_id(text):
    m = re.search(r"(blk_-?\d+)", str(text))
    return m.group(1) if m else None


def normalize_text(text):
    text = str(text).strip()
    text = re.sub(r"0x[0-9a-fA-F]+", " HEX ", text)
    text = re.sub(r"(?:\d{1,3}\.){3}\d{1,3}", " IP ", text)
    text = re.sub(r"blk_-?\d+", " BLOCKID ", text)
    text = re.sub(r"\d{5,}", " NUM ", text)
    text = re.sub(r"\d+", " NUM ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()


def compute_metrics(y_true, y_proba, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba).astype(float)
    y_pred = (y_proba >= threshold).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    for name, func in {
        "auc_roc": lambda: roc_auc_score(y_true, y_proba),
        "auc_pr": lambda: average_precision_score(y_true, y_proba),
        "brier": lambda: brier_score_loss(y_true, y_proba),
    }.items():
        try:
            out[name] = func()
        except Exception:
            out[name] = np.nan
    return out


from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig


def build_template_miner():
    config = TemplateMinerConfig()
    config.profiling_enabled = False
    config.drain_depth = 4
    config.drain_sim_th = 0.4
    config.drain_max_children = 100
    config.parametrize_numeric_tokens = True
    return TemplateMiner(config=config)


def parse_templates_for_instances(df, text_col="messages"):
    miner = build_template_miner()
    templates_out = []
    for msgs in tqdm(df[text_col].tolist(), desc="Drain3 parse"):
        seq = []
        for msg in msgs:
            msg_norm = normalize_text(msg)
            result = miner.add_log_message(msg_norm)
            if result is not None and result.get("template_mined") is not None:
                tpl = result["template_mined"]
            else:
                tpl = msg_norm
            seq.append(tpl)
        templates_out.append(seq)
    df = df.copy()
    df["templates"] = templates_out
    df["sequence_text"] = [" [SEP] ".join(x) for x in templates_out]
    return df


def load_hdfs_labels(label_csv_path):
    df = pd.read_csv(label_csv_path)
    block_col, label_col = None, None
    for c in df.columns:
        cl = c.lower()
        if "block" in cl:
            block_col = c
        if "label" in cl:
            label_col = c
    if block_col is None:
        block_col = df.columns[0]
    if label_col is None:
        label_col = df.columns[1]
    out = df[[block_col, label_col]].copy()
    out.columns = ["block_id", "label_raw"]
    out["label"] = out["label_raw"].astype(str).str.lower().map(
        lambda x: 1 if ("anomaly" in x or "abnormal" in x or x == "1") else 0
    )
    out["block_id"] = out["block_id"].astype(str).str.strip()
    return out[["block_id", "label"]]


def build_hdfs_sessions(hdfs_log_path, hdfs_label_path):
    labels_df = load_hdfs_labels(hdfs_label_path)
    label_map = dict(zip(labels_df["block_id"], labels_df["label"]))
    groups = {}
    with open(hdfs_log_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, desc="Lecture HDFS"):
            line = line.rstrip("
")
            block_id = extract_block_id(line)
            if block_id is None or block_id not in label_map:
                continue
            groups.setdefault(block_id, []).append(line)
    rows = []
    for block_id, msgs in groups.items():
        rows.append({
            "sample_id": f"HDFS::{block_id}",
            "dataset": "HDFS",
            "group_id": block_id,
            "messages": msgs,
            "label": int(label_map[block_id]),
        })
    return pd.DataFrame(rows)


def build_bgl_windows(bgl_log_path, window_size=40, stride=20):
    lines, labels = [], []
    with open(bgl_log_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in tqdm(f, desc="Lecture BGL"):
            raw = raw.rstrip("
")
            if not raw.strip():
                continue
            first = raw.split()[0]
            label = 0 if first == "-" else 1
            lines.append(raw)
            labels.append(label)
    rows = []
    for start in tqdm(range(0, max(0, len(lines) - window_size + 1), stride), desc="Fenêtres BGL"):
        end = start + window_size
        win_lines = lines[start:end]
        win_labels = labels[start:end]
        rows.append({
            "sample_id": f"BGL::{start}_{end}",
            "dataset": "BGL",
            "group_id": f"{start}_{end}",
            "messages": win_lines,
            "label": int(max(win_labels)),
        })
    return pd.DataFrame(rows)


ANOMALY_TERMS = [
    "error", "exception", "fatal", "fail", "failed", "warn", "warning",
    "abort", "panic", "timeout", "timed out", "unable", "refused"
]
ANOMALY_PATTERN = re.compile("|".join(re.escape(x) for x in ANOMALY_TERMS), re.IGNORECASE)


def add_sequence_features(df):
    df = df.copy()
    df["seq_len_raw"] = df["messages"].apply(len)
    df["n_templates"] = df["templates"].apply(len)
    df["n_unique_templates"] = df["templates"].apply(lambda x: len(set(x)))
    df["anomaly_keyword_count"] = df["sequence_text"].apply(lambda x: len(ANOMALY_PATTERN.findall(x)))
    df["anomaly_keyword_ratio"] = df["anomaly_keyword_count"] / df["n_templates"].clip(lower=1)
    df["avg_template_len"] = df["templates"].apply(
        lambda seq: float(np.mean([len(t.split()) for t in seq])) if len(seq) else 0.0
    )
    return df


STAT_COLS = [
    "seq_len_raw", "n_templates", "n_unique_templates",
    "anomaly_keyword_count", "anomaly_keyword_ratio", "avg_template_len"
]


class FeatureBuilder:
    def __init__(self, model_family="lr"):
        self.model_family = model_family
        self.vectorizer = TfidfVectorizer(
            max_features=TFIDF_MAX_FEATURES,
            ngram_range=TFIDF_NGRAM_RANGE,
            min_df=TFIDF_MIN_DF,
            lowercase=False,
        )
        self.scaler = StandardScaler()
        self.svd = None

    def fit_transform(self, df):
        X_text = self.vectorizer.fit_transform(df["sequence_text"].tolist())
        X_stats = self.scaler.fit_transform(df[STAT_COLS].values)
        if self.model_family in {"rf", "xgb"}:
            n_comp = min(TEXT_SVD_COMPONENTS, max(2, X_text.shape[1] - 1))
            self.svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
            X_text = self.svd.fit_transform(X_text)
            return np.hstack([X_text, X_stats])
        return sparse.hstack([X_text, sparse.csr_matrix(X_stats)], format="csr")

    def transform(self, df):
        X_text = self.vectorizer.transform(df["sequence_text"].tolist())
        X_stats = self.scaler.transform(df[STAT_COLS].values)
        if self.model_family in {"rf", "xgb"}:
            X_text = self.svd.transform(X_text)
            return np.hstack([X_text, X_stats])
        return sparse.hstack([X_text, sparse.csr_matrix(X_stats)], format="csr")


def make_base_model(family, y_train=None):
    family = family.lower()
    if family == "lr":
        return LogisticRegression(max_iter=3000, solver="liblinear", random_state=SEED, class_weight="balanced")
    if family == "svm":
        base = LinearSVC(C=1.0, random_state=SEED, class_weight="balanced")
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    if family == "rf":
        return RandomForestClassifier(
            n_estimators=350, max_depth=None, min_samples_split=2,
            min_samples_leaf=1, n_jobs=-1, random_state=SEED,
            class_weight="balanced_subsample",
        )
    if family == "xgb":
        spw = compute_scale_pos_weight(y_train) if y_train is not None else 1.0
        return XGBClassifier(
            n_estimators=350, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            objective="binary:logistic", eval_metric="logloss",
            random_state=SEED, n_jobs=-1, scale_pos_weight=spw,
        )
    raise ValueError(f"Famille inconnue: {family}")


def predict_proba_1(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = np.asarray(scores, dtype=float)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
        return scores
    pred = model.predict(X)
    return np.asarray(pred, dtype=float)


def make_fusion_model(family, y_train=None):
    family = family.lower()
    if family == "lr":
        return LogisticRegression(max_iter=3000, solver="liblinear", random_state=SEED, class_weight="balanced")
    if family == "svm":
        base = LinearSVC(C=1.0, random_state=SEED, class_weight="balanced")
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    if family == "rf":
        return RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED, class_weight="balanced_subsample")
    if family == "xgb":
        spw = compute_scale_pos_weight(y_train) if y_train is not None else 1.0
        return XGBClassifier(
            n_estimators=250, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            objective="binary:logistic", eval_metric="logloss",
            random_state=SEED, n_jobs=-1, scale_pos_weight=spw,
        )
    raise ValueError(f"Famille fusion inconnue: {family}")


def build_fusion_matrix(df, p_hdfs, p_bgl, include_domain_flag=True):
    X = df[STAT_COLS].copy().reset_index(drop=True)
    X.insert(0, "p_hdfs", np.asarray(p_hdfs))
    X.insert(1, "p_bgl", np.asarray(p_bgl))
    if include_domain_flag:
        X["domain_flag"] = (df["dataset"].values == "BGL").astype(int)
    return X


def get_oof_predictions(model_factory, X, y, n_splits=5):
    y = np.asarray(y).astype(int)
    oof = np.zeros(len(y), dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for tr_idx, va_idx in skf.split(np.zeros(len(y)), y):
        model = model_factory(y[tr_idx])
        model.fit(X[tr_idx], y[tr_idx])
        oof[va_idx] = predict_proba_1(model, X[va_idx])
    return oof


def plot_curves_and_cm(y_true, y_proba, title_prefix="Model"):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba).astype(float)
    y_pred = (y_proba >= 0.5).astype(int)

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f"ROC AUC={roc_auc_score(y_true, y_proba):.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC — {title_prefix}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(rec, prec, label=f"PR AUC={average_precision_score(y_true, y_proba):.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"PR Curve — {title_prefix}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(4, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix — {title_prefix}")
    plt.colorbar()
    ticks = np.arange(2)
    plt.xticks(ticks, ["Normal", "Anomaly"])
    plt.yticks(ticks, ["Normal", "Anomaly"])
    thresh = cm.max() / 2.0 if cm.max() else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()


def evaluate_pipeline_family(family, hdfs_train, hdfs_test, bgl_train, bgl_test):
    print(f"
================ {family.upper()} ================")

    fb_hdfs = FeatureBuilder(family)
    Xh_tr = fb_hdfs.fit_transform(hdfs_train)
    Xh_te = fb_hdfs.transform(hdfs_test)
    yh_tr = hdfs_train["label"].values.astype(int)
    yh_te = hdfs_test["label"].values.astype(int)

    fb_bgl = FeatureBuilder(family)
    Xb_tr = fb_bgl.fit_transform(bgl_train)
    Xb_te = fb_bgl.transform(bgl_test)
    yb_tr = bgl_train["label"].values.astype(int)
    yb_te = bgl_test["label"].values.astype(int)

    model_hdfs = make_base_model(family, yh_tr)
    model_bgl = make_base_model(family, yb_tr)
    model_hdfs.fit(Xh_tr, yh_tr)
    model_bgl.fit(Xb_tr, yb_tr)

    ph_hdfs_on_hdfs = predict_proba_1(model_hdfs, Xh_te)
    pb_bgl_on_bgl = predict_proba_1(model_bgl, Xb_te)
    ph_hdfs_on_bgl = predict_proba_1(model_hdfs, fb_hdfs.transform(bgl_test))
    pb_bgl_on_hdfs = predict_proba_1(model_bgl, fb_bgl.transform(hdfs_test))

    oof_hdfs = get_oof_predictions(lambda y: make_base_model(family, y), Xh_tr, yh_tr, n_splits=N_SPLITS_OOF)
    oof_bgl = get_oof_predictions(lambda y: make_base_model(family, y), Xb_tr, yb_tr, n_splits=N_SPLITS_OOF)

    p_bgl_on_hdfs_train = predict_proba_1(model_bgl, fb_bgl.transform(hdfs_train))
    p_hdfs_on_bgl_train = predict_proba_1(model_hdfs, fb_hdfs.transform(bgl_train))

    fusion_train_hdfs = build_fusion_matrix(hdfs_train, oof_hdfs, p_bgl_on_hdfs_train, include_domain_flag=True)
    fusion_train_bgl = build_fusion_matrix(bgl_train, p_hdfs_on_bgl_train, oof_bgl, include_domain_flag=True)
    fusion_train = pd.concat([fusion_train_hdfs, fusion_train_bgl], axis=0).reset_index(drop=True)
    y_fusion_train = np.concatenate([yh_tr, yb_tr])

    fusion_test_hdfs = build_fusion_matrix(hdfs_test, ph_hdfs_on_hdfs, pb_bgl_on_hdfs, include_domain_flag=True)
    fusion_test_bgl = build_fusion_matrix(bgl_test, ph_hdfs_on_bgl, pb_bgl_on_bgl, include_domain_flag=True)
    fusion_test_all = pd.concat([fusion_test_hdfs, fusion_test_bgl], axis=0).reset_index(drop=True)
    y_fusion_test = np.concatenate([yh_te, yb_te])

    scaler_fusion = StandardScaler()
    X_fusion_train = scaler_fusion.fit_transform(fusion_train.values)
    X_fusion_test_hdfs = scaler_fusion.transform(fusion_test_hdfs.values)
    X_fusion_test_bgl = scaler_fusion.transform(fusion_test_bgl.values)
    X_fusion_test_all = scaler_fusion.transform(fusion_test_all.values)

    fusion_model = make_fusion_model(family, y_fusion_train)
    fusion_model.fit(X_fusion_train, y_fusion_train)

    pf_hdfs = predict_proba_1(fusion_model, X_fusion_test_hdfs)
    pf_bgl = predict_proba_1(fusion_model, X_fusion_test_bgl)
    pf_all = predict_proba_1(fusion_model, X_fusion_test_all)

    rows = []
    evals = [
        (f"{family.upper()}_HDFS_expert_on_HDFS", yh_te, ph_hdfs_on_hdfs),
        (f"{family.upper()}_BGL_expert_on_BGL", yb_te, pb_bgl_on_bgl),
        (f"{family.upper()}_HDFS_expert_on_BGL", yb_te, ph_hdfs_on_bgl),
        (f"{family.upper()}_BGL_expert_on_HDFS", yh_te, pb_bgl_on_hdfs),
        (f"{family.upper()}_Fusion_on_HDFS", yh_te, pf_hdfs),
        (f"{family.upper()}_Fusion_on_BGL", yb_te, pf_bgl),
        (f"{family.upper()}_Fusion_on_Combined", y_fusion_test, pf_all),
    ]
    for name, y_true, y_proba in evals:
        row = {"model": name}
        row.update(compute_metrics(y_true, y_proba))
        rows.append(row)

    return pd.DataFrame(rows), {
        "fusion_hdfs": (yh_te, pf_hdfs),
        "fusion_bgl": (yb_te, pf_bgl),
        "fusion_all": (y_fusion_test, pf_all),
    }


hdfs_log_path = find_file(ROOT_DIR, HDFS_LOG_NAME)
hdfs_label_path = find_file(ROOT_DIR, HDFS_LABEL_NAME)
bgl_log_path = find_file(ROOT_DIR, BGL_LOG_NAME)

print("HDFS log    :", hdfs_log_path)
print("HDFS labels :", hdfs_label_path)
print("BGL log     :", bgl_log_path)
assert hdfs_log_path is not None, f"{HDFS_LOG_NAME} introuvable sous {ROOT_DIR}"
assert hdfs_label_path is not None, f"{HDFS_LABEL_NAME} introuvable sous {ROOT_DIR}"
assert bgl_log_path is not None, f"{BGL_LOG_NAME} introuvable sous {ROOT_DIR}"

hdfs_df = build_hdfs_sessions(hdfs_log_path, hdfs_label_path)
bgl_df = build_bgl_windows(bgl_log_path, window_size=BGL_WINDOW_SIZE, stride=BGL_STRIDE)

print_distribution(hdfs_df, "HDFS brut")
print_distribution(bgl_df, "BGL brut")

hdfs_df = safe_sample_frac(hdfs_df, HDFS_SAMPLE_FRAC, label_col="label", seed=SEED)
bgl_df = safe_sample_frac(bgl_df, BGL_SAMPLE_FRAC, label_col="label", seed=SEED)
print_distribution(hdfs_df, "HDFS sample")
print_distribution(bgl_df, "BGL sample")
print("
Aucun resampling n'est appliqué. La gestion du déséquilibre se fait par class weights.")

hdfs_df = parse_templates_for_instances(hdfs_df, text_col="messages")
bgl_df = parse_templates_for_instances(bgl_df, text_col="messages")

hdfs_df = add_sequence_features(hdfs_df)
bgl_df = add_sequence_features(bgl_df)

hdfs_train, hdfs_test = train_test_split(
    hdfs_df, test_size=TEST_SIZE, random_state=SEED, stratify=hdfs_df["label"]
)
bgl_train, bgl_test = train_test_split(
    bgl_df, test_size=TEST_SIZE, random_state=SEED, stratify=bgl_df["label"]
)

hdfs_train = hdfs_train.reset_index(drop=True)
hdfs_test = hdfs_test.reset_index(drop=True)
bgl_train = bgl_train.reset_index(drop=True)
bgl_test = bgl_test.reset_index(drop=True)

print_distribution(hdfs_train, "HDFS train")
print_distribution(hdfs_test, "HDFS test")
print_distribution(bgl_train, "BGL train")
print_distribution(bgl_test, "BGL test")
print(f"Scale_pos_weight HDFS train: {compute_scale_pos_weight(hdfs_train['label'].values):.4f}")
print(f"Scale_pos_weight BGL  train: {compute_scale_pos_weight(bgl_train['label'].values):.4f}")

all_results = []
curves = {}
for family in ["lr", "svm", "rf", "xgb"]:
    df_res, curve_dict = evaluate_pipeline_family(family, hdfs_train, hdfs_test, bgl_train, bgl_test)
    all_results.append(df_res)
    curves[family] = curve_dict

results_df = pd.concat(all_results, axis=0).reset_index(drop=True)
results_df = results_df.sort_values(["f1", "auc_pr", "auc_roc"], ascending=False).reset_index(drop=True)
print("
===== Tableau comparatif global =====")
print(results_df.round(4))

for family in ["lr", "svm", "rf", "xgb"]:
    print(f"
--- Courbes et matrices : {family.upper()} Fusion on Combined ---")
    y_true, y_proba = curves[family]["fusion_all"]
    plot_curves_and_cm(y_true, y_proba, title_prefix=f"{family.upper()} Fusion Combined")
